# Olist Reviews Agent

Foco exclusivo nas avaliações negativas para identificar reclamações de atraso, defeito, item faltando e qualidade da experiência.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

ModuleNotFoundError: No module named 'pandas'

## Carregar dados relevantes de reviews

Será usada a tabela de reviews e os itens/pedidos relacionados para cruzar sellers e categorias.

In [2]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('reviews', reviews.shape)
print('order_items', order_items.shape)
print('products', products.shape)
print('sellers', sellers.shape)

NameError: name 'pd' is not defined

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('reviews', reviews.shape)
print('order_items', order_items.shape)
print('products', products.shape)
print('sellers', sellers.shape)

## Análise de avaliações negativas

Filtrar reviews com nota menor que 4 e identificar padrões de reclamação.

In [ ]:
negative_reviews = reviews[reviews['review_score'] < 4].copy()
negative_reviews['review_text'] = negative_reviews['review_comment_message'].fillna('').str.lower()

plt.figure(figsize=(8, 5))
score_counts = negative_reviews['review_score'].value_counts().sort_index()
sns.barplot(x=score_counts.index, y=score_counts.values, palette='Reds')
plt.title('Distribuição das avaliações negativas (score < 4)')
plt.xlabel('Review score')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

### Classificação de tipos de reclamação

Agrupar palavras-chave em categorias como atraso, defeito e faltando.

In [ ]:
keywords = {
    'atraso': ['atraso', 'atrasou', 'entrega atrasada', 'demora', 'entregue tarde', 'late', 'não chegou'],
    'defeito': ['defeito', 'quebrado', 'danificado', 'estragado', 'avaria', 'não funciona', 'funcionando'],
    'faltando': ['faltando', 'falta', 'não veio', 'ausente', 'faltou', 'incompleto'],
    'produto errado': ['produto errado', 'errado', 'não era', 'não corresponde', 'item errado'],
    'qualidade ruim': ['qualidade ruim', 'ruim', 'pior', 'péssimo', 'decepcionante'],
}

def classify_review(text):
    for label, terms in keywords.items():
        if any(term in text for term in terms):
            return label
    return 'outros'

negative_reviews['issue_type'] = negative_reviews['review_text'].apply(classify_review)

issue_counts = negative_reviews['issue_type'].value_counts().reset_index()
issue_counts.columns = ['issue_type', 'count']

plt.figure(figsize=(10, 5))
sns.barplot(data=issue_counts, x='count', y='issue_type', palette='rocket')
plt.title('Tipos de reclamação em avaliações negativas')
plt.xlabel('Quantidade')
plt.ylabel('Tipo de reclamação')
plt.tight_layout()
plt.show()

## Cruzamento com sellers e categorias

Associar as avaliações negativas a sellers e produtos para identificar áreas problemáticas.

In [ ]:
merged_reviews = pd.merge(negative_reviews, order_items, on='order_id', how='left')
merged_reviews = pd.merge(merged_reviews, products[['product_id', 'product_category_name']], on='product_id', how='left')
merged_reviews = pd.merge(merged_reviews, sellers[['seller_id', 'seller_label']], on='seller_id', how='left')

seller_issues = (
    merged_reviews.groupby(['seller_label', 'issue_type'])
    .size()
    .unstack(fill_value=0)
)
seller_issues['total_negative'] = seller_issues.sum(axis=1)

top_sellers_negative = seller_issues.sort_values('total_negative', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_sellers_negative['total_negative'], y=top_sellers_negative.index, palette='magma')
plt.title('Top 10 sellers com mais avaliações negativas')
plt.xlabel('Total de avaliações negativas')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

## Categorias com mais reclamações negativas

In [ ]:
category_issues = (
    merged_reviews.groupby('product_category_name')['issue_type']
    .count()
    .reset_index(name='negative_reviews')
    .sort_values('negative_reviews', ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 6))
sns.barplot(data=category_issues, x='negative_reviews', y='product_category_name', palette='viridis')
plt.title('Top 10 categorias com mais avaliações negativas')
plt.xlabel('Quantidade de avaliações negativas')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()

## Exemplo de reviews negativas classificadas

In [ ]:
negative_reviews[['review_score', 'issue_type', 'review_text']].head(10)